# YZTA Datathon — Keşifsel Veri Analizi (EDA)

Bu notebook sadece keşif ve görselleştirme içindir. Üretim kodu `src/`'de.
Bulgular `src/features.py` ve `src/data.py`'ye yansıtılır.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_and_preprocess
from src.config import TARGET, ID_COL, CATEGORICAL_COLS, NUMERIC_COLS

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)

train, test, sample_sub = load_and_preprocess()
print(f'Train: {train.shape}, Test: {test.shape}')

## 1. Hedef Değişken Dağılımı

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(train[TARGET], bins=60, color='#4C72B0', edgecolor='white')
axes[0].set_title('Bilişsel Performans Skoru — Histogram')
axes[0].set_xlabel(TARGET)
axes[1].boxplot(train[TARGET], vert=False)
axes[1].set_title('Boxplot')
plt.tight_layout(); plt.show()

print(train[TARGET].describe().round(3))
print(f'Skewness: {train[TARGET].skew():.3f}')
print(f'Kurtosis: {train[TARGET].kurt():.3f}')

## 2. Eksik Değerler

In [ ]:
missing = (train.isnull().sum()
    .to_frame('count')
    .assign(pct=lambda d: 100*d['count']/len(train))
    .query('count > 0').sort_values('count', ascending=False))
print(missing)

if not missing.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    missing['pct'].plot(kind='barh', ax=ax, color='#DD8452')
    ax.set_title('Eksik Değer Yüzdeleri')
    plt.tight_layout(); plt.show()

## 3. Korelasyon Matrisi

In [ ]:
numeric_cols_full = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_full.remove(ID_COL)
corr = train[numeric_cols_full].corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            annot_kws={'size': 7}, cbar_kws={'shrink': 0.7}, ax=ax)
ax.set_title('Pearson Korelasyon Matrisi')
plt.tight_layout(); plt.show()

target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print('Hedefle en güçlü korelasyonlar:')
print(target_corr.round(3))

## 4. Kategorik Değişkenler → Hedef

In [ ]:
for col in CATEGORICAL_COLS:
    print(f'--- {col} ---')
    print(train.groupby(col, observed=True)[TARGET]
          .agg(['mean','std','count']).round(3))
    print()

In [ ]:
key_cats = ['kronotip','ruh_sagligi_durumu','cinsiyet','gun_tipi']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flatten(), key_cats):
    sns.boxplot(data=train, x=col, y=TARGET, ax=ax)
    ax.set_title(f'{col} → {TARGET}')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 5. Ülke Sütunu Kontrolü

EDA bulgusu: 3 ülke iki kez kodlanmış (Spain/Ispanya, South Korea/Guney Kore, Sweden/Isvec).
`src/data.py:normalize_country` bunu birleştiriyor.

In [ ]:
print('Train ülkeler (preprocess sonrası):')
print(train['ulke'].value_counts())
print(f'\nUnique sayısı: {train["ulke"].nunique()}')